<a href="https://colab.research.google.com/github/viettungvuong/Distillation-Playground/blob/main/Distillation_Playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 152.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


**Huggingface login**

In [19]:
from huggingface_hub import login

login()

**Downloading dataset**

In [22]:
from datasets import load_dataset

dataset = load_dataset("OpenHust/vietnamese-summarization")

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

model_name = "Qwen/Qwen3.6-27B"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.31k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

**Data processing**

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Document', 'Summary', 'Dataset'],
        num_rows: 74564
    })
})

In [5]:
split_dataset = dataset["train"].train_test_split(test_size=0.2)

train_set = split_dataset['train']
test_set = split_dataset['test']

In [6]:
train_set[:5]

{'Unnamed: 0': [7081, None, 3503, None, 1211],
 'Document': ['Việc N quyết định V tẩy V lông N ở nơi N nào trước N sẽ tùy V thuộc V vào chính bạn N , tuy nhiên bạn N cần V áp dụng V theo V trình tự N khi N tiến hành V tẩy V lông N thành V từng mảng N . Ví dụ N , bạn N có thể bắt đầu V tẩy V lông N từ phần N bắp đùi N bên trong N và di chuyển V về phía N háng N , sau N đó kết thúc V ở vùng N hậu môn N . Tay trái N nhẹ nhàng A kéo căng N vùng N da N mà bạn N dự định V tẩy N đầu tiên A . Dùng V que N lấy V một ít sáp N ấm A và phết V nhẹ A lên V phần N da N cần V tẩy V lông N trước tiên N . Cách N này sẽ giúp V quá trình N tẩy V lông N trở nên V dễ dàng A và ít A đau đớn A hơn A . Thoa một ít sáp N ở chỗ N lông N không mọc V nhằm V tạo V thành V một " dải N " mà bạn N có thể nắm V lấy để kéo V miếng N sáp V ra V . Sáp N sẽ cứng V lại V sau N khi N đã nguội V hoàn toàn A . Khi N sáp N cứng A lại V , bạn N sẽ kéo V sáp V ra V dễ dàng A hơn A . Dùng tay N phải V nắm V lấy V mép N cuối N " dả

In [7]:
def normalize_text(row):
    # Only handle lowercase normalization
    row['Document'] = str(row['Document']).lower()
    row['Summary'] = str(row['Summary']).lower()
    return row

def add_special_tokens(row):
    # Qwen follows the ChatML format:
    # <|im_start|>user
    # {Document}<|im_end|>
    # <|im_start|>assistant
    # {Summary}<|im_end|>

    formatted_input = (
        f"<|im_start|>user\n"
        f"{row['Document']}<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"{row['Summary']}<|im_end|>"
    )

    row['Processed_Input'] = formatted_input
    return row

# Apply normalization
train_set = train_set.map(normalize_text)
test_set = test_set.map(normalize_text)

# Apply special tokens and create new column
train_set = train_set.map(add_special_tokens)
test_set = test_set.map(add_special_tokens)

Map:   0%|          | 0/59651 [00:00<?, ? examples/s]

Map:   0%|          | 0/14913 [00:00<?, ? examples/s]

Map:   0%|          | 0/59651 [00:00<?, ? examples/s]

Map:   0%|          | 0/14913 [00:00<?, ? examples/s]

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [9]:
print("Special tokens: ", tokenizer.special_tokens_map)
sep_token_id = tokenizer.convert_tokens_to_ids("<|turn>")
print(f"Sep token is {sep_token_id}")

Special tokens:  {'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'mask_token': '<mask>', 'audio_token': '<|audio|>', 'boa_token': '<|audio>', 'boi_token': '<|image>', 'eoa_token': '<audio|>', 'eoc_token': '<channel|>', 'eoi_token': '<image|>', 'eot_token': '<turn|>', 'escape_token': '<|"|>', 'etc_token': '<tool_call|>', 'etd_token': '<tool|>', 'etr_token': '<tool_response|>', 'image_token': '<|image|>', 'soc_token': '<|channel>', 'sot_token': '<|turn>', 'stc_token': '<|tool_call>', 'std_token': '<|tool>', 'str_token': '<|tool_response>', 'think_token': '<|think|>'}
Sep token is 105


In [10]:
def tokenize_function(batch):
    # Tokenize the processed input
    model_inputs = tokenizer(
        batch["Processed_Input"],
        truncation=True,
        max_length=1542,
        padding='max_length'
    )

    labels = []

    for input_ids in model_inputs["input_ids"]:
        label = list(input_ids)
        try:
            # Find the index of the first separation token
            indices = [i for i, val in enumerate(label) if val == sep_token_id]
            turn_model_idx = indices[-1]
            # Set all tokens up to the first separation to -100 (loss function ignore)
            for i in range(turn_model_idx + 1):
                label[i] = -100
        except ValueError as e:
            print(e)

        labels.append(label)

    model_inputs["labels"] = labels
    return model_inputs


In [11]:
tokenized_train = train_set.map(tokenize_function, batched=True, remove_columns=train_set.column_names)
tokenized_test = test_set.map(tokenize_function, batched=True, remove_columns=test_set.column_names)

Map:   0%|          | 0/59651 [00:00<?, ? examples/s]

Map:   0%|          | 0/14913 [00:00<?, ? examples/s]

**Fine-tuning**

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="summarization",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

In [17]:
from transformers import Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [18]:
trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 6.02 GiB. GPU 0 has a total capacity of 79.25 GiB of which 4.29 GiB is free. Including non-PyTorch memory, this process has 74.95 GiB memory in use. Of the allocated memory 65.05 GiB is allocated by PyTorch, and 9.40 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)